# Real-World Routing with OSRM

GPS coordinates, road-network travel times and distances, and map plots.

Two OSRM options:

- **Remote server** — zero setup, fine for small instances (max ~50 locations
  on the public demo server).
- **Local Docker** — `OSRMLocalManager` downloads the map region, preprocesses
  it, and runs a containerized OSRM. Needs Docker; data is cached in
  `~/.cache/tqrouting/osrm`.

Docs: Distance & Duration Matrices page.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from tqrouting import TQRouter, VRPInstance, Customer, VehicleType, Location, TimeWindow
from tqrouting.matrix import EuclideanMatrixProvider

data_dir = Path("data")

# Uses the TQROUTING_LICENSE_KEY environment variable by default.
# Set it before running: export TQROUTING_LICENSE_KEY="..."
router = TQRouter()

## 1. Remote OSRM — lightweight example

A few locations in Munich against the public demo server.

In [ ]:
from tqrouting.matrix import OSRMMatrixProvider

instance_osrm = VRPInstance(
    customers=[
        Customer(id="Marienplatz", location=Location(lat=48.1374, lon=11.5755), delivery_load=5),
        Customer(id="Olympiapark", location=Location(lat=48.1735, lon=11.5461), delivery_load=3),
        Customer(id="Deutsches Museum", location=Location(lat=48.1298, lon=11.5835), delivery_load=4),
        Customer(id="BMW Welt", location=Location(lat=48.1770, lon=11.5562), delivery_load=6),
        Customer(id="Nymphenburg", location=Location(lat=48.1583, lon=11.5035), delivery_load=2),
    ],
    fleet=[
        VehicleType(
            start_location=Location(lat=48.1401, lon=11.5600),
            end_location=Location(lat=48.1401, lon=11.5600),
            capacity_load=15,
            n_vehicles=3,
        ),
    ],
)

# Public demo server - suitable for testing only.
provider = OSRMMatrixProvider(base_url="https://router.project-osrm.org")
instance_osrm.compute_duration_matrix(provider)   # seconds on the road network
instance_osrm.compute_distance_matrix(provider)   # meters on the road network

sol_osrm = router.solve(instance_osrm, time_limit=3)
for r in sol_osrm.routes:
    stops = [v.customer_id for v in r.visits]
    print(f"Route {r.id}: {stops} ({r.route_duration:.0f}s)")

With both matrices in place you can also price distance into the objective —
see [objectives.ipynb](objectives.ipynb):

```python
router.solve(instance_osrm, distance_weight=0.1)   # 1 km ~ 100 s
```

## 2. Bayern — 500 customers, local OSRM (Docker)

A synthetic delivery instance on real road geometry: lat/lon locations around
Ingolstadt, `HH:MM:SS` time windows, heterogeneous fleet with distance limits.

**Requires Docker.** The first run downloads and preprocesses the Oberbayern
map extract (a few minutes); later runs reuse the cache.

In [ ]:
instance_bayern = VRPInstance.from_json(data_dir / "bayern_instance.json")

print(f"Bayern: {len(instance_bayern.customers)} customers, "
      f"{len(instance_bayern.fleet)} vehicle types")
for vt in instance_bayern.fleet:
    print(f"  {vt.id}: capacity {vt.capacity_load}, n={vt.n_vehicles}, "
          f"max_duration={vt.max_duration}"
          + (f", max_distance={vt.max_distance}" if vt.max_distance else ""))

In [ ]:
from tqrouting.matrix import OSRMLocalManager

with OSRMLocalManager(region="europe/germany/bayern/oberbayern", profile="car") as osrm:
    instance_bayern.compute_distance_matrix(osrm)
    instance_bayern.compute_duration_matrix(osrm)

n = len(instance_bayern.distance_matrix)
print(f"Matrices computed: {n}x{n}")
print(f"  sample distance [0->1]: {instance_bayern.distance_matrix[0][1]:.0f} m")
print(f"  sample duration [0->1]: {instance_bayern.duration_matrix[0][1]:.0f} s")

## 3. Solve

In [ ]:
solution_bayern = router.solve(instance_bayern, time_limit=30, nb_threads=8)

print(f"Status: {solution_bayern.solution_status}")
print(f"Routes used: {len(solution_bayern.routes)}")
print(f"Total visits: {solution_bayern.total_n_visits} / {len(instance_bayern.customers)}")
print(f"Total driving time: {solution_bayern.total_driving_time:.0f} s")
for route in solution_bayern.routes[:8]:
    print(f"  Route {route.id} (type {route.vehicle_type_id}): "
          f"{len(route.visits)} customers, "
          f"duration={route.route_duration:.0f}s")
if len(solution_bayern.routes) > 8:
    print(f"  ... and {len(solution_bayern.routes) - 8} more routes")

## 4. Plot on a lat/lon map

In [ ]:
def plot_geo_solution(instance: VRPInstance, solution, title="Routes", max_routes=15):
    """Plot routes on a lat/lon scatter."""
    colors = plt.cm.tab20.colors
    fig, ax = plt.subplots(1, 1, figsize=(11, 8))

    id_to_loc = {}
    for i, c in enumerate(instance.customers):
        cid = c.id if c.id is not None else i
        id_to_loc[cid] = c.location

    depot = instance.fleet[0].start_location
    ax.plot(depot.lon, depot.lat, "k^", markersize=14, zorder=10, label="Depot")

    visited = {v.customer_id for r in solution.routes for v in r.visits}
    for cid, loc in id_to_loc.items():
        if cid not in visited:
            ax.plot(loc.lon, loc.lat, ".", color="lightgrey", markersize=4)

    for route in solution.routes[:max_routes]:
        color = colors[route.id % len(colors)]
        lons = [depot.lon] + [id_to_loc[v.customer_id].lon for v in route.visits] + [depot.lon]
        lats = [depot.lat] + [id_to_loc[v.customer_id].lat for v in route.visits] + [depot.lat]
        ax.plot(lons, lats, "o-", color=color, markersize=3, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    plt.show()

plot_geo_solution(instance_bayern, solution_bayern, title="Bayern - 500 customers")